# GCS 버킷 구조 조회

FAISS 인덱스 원본 버킷의 폴더/파일 구조를 JSON으로 출력한다.

전제: `gcloud auth login` 완료 (`upload_vectordb.sh`와 같은 자격 증명).

In [1]:
import json
import subprocess

BUCKET = "gs://project-1bbc94dc-a155-4b6b-8a5-vectordb"

# -r --long: "크기 수정시각 URL" 3열 + 마지막 TOTAL 행
listing = subprocess.run(
    ["gcloud", "storage", "ls", "-r", "--long", f"{BUCKET}/**"],
    capture_output=True,
    text=True,
    check=True,
).stdout

tree: dict = {}
for line in listing.splitlines():
    parts = line.split()
    if len(parts) != 3 or not parts[2].startswith(BUCKET):  # TOTAL 행 제외
        continue
    size, updated, url = parts
    *folders, name = url[len(BUCKET) + 1 :].split("/")
    node = tree
    for folder in folders:
        node = node.setdefault(folder, {})
    node[name] = {"size": int(size), "updated": updated}

print(json.dumps({BUCKET: tree}, indent=2, ensure_ascii=False))

{
  "gs://project-1bbc94dc-a155-4b6b-8a5-vectordb": {
    "laws_faiss": {
      "documents.jsonl": {
        "size": 4138650,
        "updated": "2026-09-19T14:08:16Z"
      },
      "id_map.json": {
        "size": 99582,
        "updated": "2026-09-19T14:08:14Z"
      },
      "index.faiss": {
        "size": 37625901,
        "updated": "2026-09-19T14:08:26Z"
      },
      "index.pkl": {
        "size": 3292397,
        "updated": "2026-09-19T14:08:16Z"
      },
      "manifest.json": {
        "size": 464,
        "updated": "2026-09-19T14:08:14Z"
      }
    },
    "precedents_faiss": {
      "documents.jsonl": {
        "size": 2731114,
        "updated": "2026-09-19T14:08:33Z"
      },
      "id_map.json": {
        "size": 41386,
        "updated": "2026-09-19T14:08:32Z"
      },
      "index.faiss": {
        "size": 15986733,
        "updated": "2026-09-19T14:08:37Z"
      },
      "index.pkl": {
        "size": 2284952,
        "updated": "2026-09-19T14:08:33Z"
      },
   